# Selection — Solutions

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy  as np
import tables as tb
import pandas as pd
import matplotlib.pyplot as plt

import scipy.constants as constants

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, sys
from pathlib import Path

# Find the project root: honours FANAL_ROOT env-var, otherwise walks up from cwd
_env = os.environ.get('FANAL_ROOT') or os.environ.get('USCFANALDIR')
if _env and Path(_env, 'data').is_dir():
    rootpath = str(Path(_env).resolve())
else:
    rootpath = str(next(p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'data').is_dir() and (p / 'ana').is_dir()))
if rootpath not in sys.path:
    sys.path.insert(0, rootpath)
print('Fanal root : ', rootpath)

In [ ]:
import core.pltext  as pltext   # extensions for plotting histograms
import core.hfit    as hfit     # extension to fit histograms
import core.utils   as ut       # generic utilities
import ana.fanal    as fn       # analysis functions specific to fanal
pltext.style()

In [ ]:
coll          = 'new_gamma'
sel_ntracks  = 1
sel_eblob2   = 0.400 # MeV
sel_erange   = ( 2.400,  2.700) # MeV
sel_eroi     = ( 2.430,  2.480) # MeV

print('Collaboration             : {:s}'.format(coll))
print('number of tracks range    : {:6d}'.format(sel_ntracks))
print('Blob-2 energy range       : {:6.3f}  MeV'.format(sel_eblob2))
print('Energy range              : ({:6.3f}, {:6.3f}) MeV'.format(*sel_erange))
print('Energy RoI range          : ({:6.3f}, {:6.3f}) MeV'.format(*sel_eroi))

In [ ]:
# set the path to the data directory and filenames
dirpath = rootpath+'/data/'
filename = 'fanal_' + coll + '.h5'
print('Data path and filename : ', dirpath + filename)

# access the simulated data (DataFrames) for the different samples (Bi, Tl, bb) located in the data file
mcbi = pd.read_hdf(dirpath + filename, key = 'mc/bi214').fillna(0.)
mctl = pd.read_hdf(dirpath + filename, key = 'mc/tl208').fillna(0.)
mcbb = pd.read_hdf(dirpath + filename, key = 'mc/bb0nu').fillna(0.)

# set the names of the samples
mc_samples         = [mcbi, mctl, mcbb] # list of the mc DFs
sample_names       = ['Bi', 'Tl', 'bb']
sample_names_latex = [ r'$^{214}$Bi', r'$^{208}$Tl', r'$\beta\beta0\nu$',] # str names of the mc samples

for i, mc in enumerate(mc_samples):
    print('MC Sample {:s}, number of simulated events = {:d}'.format(sample_names[i], len(mc)))

## Compute selection efficiencies for all samples

In [ ]:
# names in the plot
sample_efficiencies = []

def efficiency(mask):
    return float(sum(mask)/len(mask))

for mc in mc_samples:

    mask1 = mc.num_tracks <= sel_ntracks
    eff1  = efficiency(mask1)
    
    mask2 = mask1 & (mc.blob2_E > sel_eblob2)
    eff2  = efficiency(mask2)
    
    mask3 = mask2 & (mc.E >= sel_erange[0]) & (mc.E < sel_erange[1])
    eff3  = efficiency(mask3)

    mask4 = mask3 & (mc.E >= sel_eroi[0]) & (mc.E < sel_eroi[1])
    eff4  = efficiency(mask4)
    
    sample_efficiencies.append((eff1, eff2, eff3, eff4))
    
sel_names  = ["track", "E blob2", "E", "RoI"]

for i, ieffs in enumerate(sample_efficiencies):
    print('Efficiencies ' + sample_names[i], ': ', \
          *[' {:s} = {:6.5f}, '.format(s, eff) for s, eff in zip(sel_names, ieffs)])

In [ ]:
subplot = pltext.canvas(1, 1, 6, 8)
subplot(1)
for sample, efficiencies in zip(sample_names_latex, sample_efficiencies):
    plt.plot(sel_names, efficiencies, marker = 'o', label = sample, lw = 2)
plt.grid(which = 'both'); plt.yscale('log'); plt.legend();
plt.title('efficiencies vs cut');